# ChEMBL Final Output Recreation

This notebook recreates the final ChEMBL summary from `chembl_stereo_clean.csv`.

Simplifications:
- The input file is always `chembl_stereo_clean.csv`.
- The worker is always loaded from the `notebooks` folder.
- Collision groups are written as CSV.
- The final result is printed as a summary instead of being formatted as LaTeX.

In [ ]:
from __future__ import annotations

import csv
import importlib.util
import multiprocessing as mp
import sys
from collections import Counter
from pathlib import Path

from tqdm.auto import tqdm


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

INPUT_CANDIDATES = [
    PROJECT_ROOT / "new_data" / "chembl_stereo_clean.csv",
    PROJECT_ROOT / "data" / "chembl_stereo_clean.csv",
]
INPUT_PATH = next((path for path in INPUT_CANDIDATES if path.exists()), None)
if INPUT_PATH is None:
    raise FileNotFoundError("Could not find chembl_stereo_clean.csv.")

OUTPUT_DIR = PROJECT_ROOT / "new_data" / "hash_collision_recreation"
SUMMARY_TSV_PATH = OUTPUT_DIR / "chembl_hash_collision_summary.tsv"
COLLISION_GROUPS_CSV_PATH = OUTPUT_DIR / "chembl_collision_groups.csv"
N_WORKERS = max(1, (mp.cpu_count() or 1) - 1)
CHUNKSIZE = 50

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Input file:    {INPUT_PATH}")
print(f"Output dir:    {OUTPUT_DIR}")
print(f"Workers:       {N_WORKERS}")

In [ ]:
worker2_spec = importlib.util.spec_from_file_location(
    "_worker2", NOTEBOOK_DIR / "_worker2.py"
 )
if worker2_spec is None or worker2_spec.loader is None:
    raise ImportError(
        f"Could not load worker module from {NOTEBOOK_DIR / '_worker2.py'}"
    )

_worker2 = importlib.util.module_from_spec(worker2_spec)
worker2_spec.loader.exec_module(_worker2)
compute_hash = _worker2.compute_hash


def iter_input_rows(input_path: Path):
    with open(input_path, encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        required = {"chembl_id", "inchi", "smiles"}
        missing = required.difference(reader.fieldnames or [])
        if missing:
            raise KeyError(f"Missing required columns: {sorted(missing)}")
        for row in reader:
            chembl_id = (row.get("chembl_id") or "").strip()
            inchi = (row.get("inchi") or "").strip()
            smiles = (row.get("smiles") or "").strip()
            if chembl_id and inchi:
                yield chembl_id, inchi, smiles


all_rows = list(iter_input_rows(INPUT_PATH))
print(f"Loaded {len(all_rows):,} input rows")

hash_counts = Counter()
hash_results = []
n_errors = 0

with mp.get_context("spawn").Pool(processes=N_WORKERS) as pool:
    for chembl_id, inchi, smg_hash, error in tqdm(
        pool.imap_unordered(compute_hash, all_rows, chunksize=CHUNKSIZE),
        total=len(all_rows),
        desc="Hashing",
        mininterval=0.3,
    ):
        if error is None and smg_hash is not None:
            hash_counts[smg_hash] += 1
            hash_results.append((chembl_id, inchi, smg_hash))
        else:
            n_errors += 1

collision_hashes = {smg_hash for smg_hash, count in hash_counts.items() if count > 1}
collision_rows = [
    (chembl_id, inchi, smg_hash)
    for chembl_id, inchi, smg_hash in hash_results
    if smg_hash in collision_hashes
]

summary = {
    "dataset": "ChEMBL",
    "molecules": len(all_rows),
    "collisions": len(collision_hashes),
    "affected_molecules": len(collision_rows),
    "affected_percent": (100.0 * len(collision_rows) / len(all_rows)) if all_rows else 0.0,
    "errors": n_errors,
}

## Write Final Outputs

This cell writes the summary TSV, stores collision-group memberships as CSV, and prints the final summary.

In [ ]:
group_sizes = {smg_hash: hash_counts[smg_hash] for smg_hash in collision_hashes}

with open(COLLISION_GROUPS_CSV_PATH, "w", encoding="utf-8", newline="") as handle:
    writer = csv.writer(handle)
    writer.writerow(["smg_hash", "group_size", "chembl_id", "inchi"])
    for chembl_id, inchi, smg_hash in sorted(
        collision_rows,
        key=lambda row: (-group_sizes[row[2]], row[2], row[0]),
    ):
        writer.writerow([smg_hash, group_sizes[smg_hash], chembl_id, inchi])

with open(SUMMARY_TSV_PATH, "w", encoding="utf-8", newline="") as handle:
    writer = csv.writer(handle, delimiter="\t")
    writer.writerow(summary.keys())
    writer.writerow(summary.values())

print("Final summary:")
for key, value in summary.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.3f}")
    elif isinstance(value, int):
        print(f"  {key}: {value:,}")
    else:
        print(f"  {key}: {value}")

print()
print(f"Wrote collision groups CSV: {COLLISION_GROUPS_CSV_PATH}")
print(f"Wrote summary TSV:          {SUMMARY_TSV_PATH}")